# 01.03 YOLOv10 推理

## 本节概述

<table style="text-align: left; margin-left: 0;">
<tr><td align="left"><b>前置要求</b></td><td align="left">已完成 01.02 OpenCV 基础</td></tr>
<tr><td align="left"><b>本节目标</b></td><td align="left">用 YOLOv10 做目标检测，解读检测结果，了解实时推理</td></tr>
<tr><td align="left"><b>本节内容</b></td><td align="left">YOLO 原理与安装 → 单图推理与结果解读 → 实时推理（摄像头/视频）</td></tr>
</table>

## 第一部分：YOLO 快速入门


In [ ]:
# 安装 ultralytics（清华源加速，约 1-2 分钟）
!pip install ultralytics -i https://pypi.tuna.tsinghua.edu.cn/simple -q

# 验证安装
from ultralytics import YOLO
import ultralytics
print("✅ Ultralytics 版本:", ultralytics.__version__)


## 1. 加载预训练模型

YOLO 模型以 `.pt` 文件（PyTorch 权重格式）分发。我们使用 **YOLOv10n**——`n` 表示 nano（最小最快版本），适合教学和快速演示。

`YOLO()` 会自动下载权重文件到当前目录：

In [ ]:
from ultralytics import YOLO

# 加载 YOLOv10n（nano 版本，最小最快）
# yolov10n.pt 首次运行时自动从 Ultralytics 下载（约 5.6MB）
model = YOLO("yolov10n.pt", task='detect')

print("✅ 模型加载完成！")
print("模型类型:", model.model_name if hasattr(model, 'model_name') else "YOLOv10n")


### 模型命名约定（n/s/m/l/x）

Ultralytics 系列按大小分 5 档，越大越准但越慢：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">后缀</th><th align="left">含义</th><th align="left">速度</th><th align="left">精度</th><th align="left">体积</th></tr>
<tr><td align="left"><code>n</code></td><td align="left">nano 纳米</td><td align="left">最快</td><td align="left">最低</td><td align="left">~5MB</td></tr>
<tr><td align="left"><code>s</code></td><td align="left">small 小</td><td align="left">快</td><td align="left">较低</td><td align="left">~20MB</td></tr>
<tr><td align="left"><code>m</code></td><td align="left">medium 中</td><td align="left">中等</td><td align="left">中</td><td align="left">~50MB</td></tr>
<tr><td align="left"><code>l</code></td><td align="left">large 大</td><td align="left">慢</td><td align="left">较高</td><td align="left">~80MB</td></tr>
<tr><td align="left"><code>x</code></td><td align="left">extra large 超大</td><td align="left">最慢</td><td align="left">最高</td><td align="left">~130MB</td></tr>
</table>

> 教学和实时应用推荐 `n` 或 `s`；追求精度选 `m` 以上。

## 2. 第一次推理：5 行代码

加载完模型，做一次推理就是这么简单：

In [ ]:
# 用一张真实照片做首次推理（YOLO 能稳定检测公交车和行人）
# bus.jpg 是 Ultralytics 经典示例图（公交车上有乘客），已随课程提供在 ./images/
import cv2
import matplotlib.pyplot as plt

# 推理：调用 model(source=...) 即可
# source 可以是图片路径、文件夹、视频、摄像头编号(0)、URL
results = model(source="./images/bus.jpg", save=True, conf=0.25)

# 可视化检测结果
annotated = results[0].plot()
plt.figure(figsize=(10, 7))
plt.imshow(annotated[:, :, ::-1])  # BGR → RGB
plt.axis("off")
plt.title("YOLO First Inference (bus.jpg)")
plt.show()

print("\n✅ 推理完成！结果保存在 runs/detect/ 目录下")
print("检测到的目标数量:", len(results[0].boxes))
print("检测到的类别:", results[0].boxes.cls.unique().tolist())

恭喜！你已经完成了第一次 YOLO 目标检测。虽然这张测试图很简单可能没有检测到明显目标，但你已经掌握了 YOLO 推理的核心调用方式。

下一节我们用真实图片，详细解读检测结果。

---

## 本节练习

**练习 1（选择）**：YOLO 的核心特点是什么？
- A. 需要多次前向传播才能检测完所有物体
- B. 只需一次前向传播就能检测出所有物体的位置和类别
- C. 只能检测一种类别的物体
- D. 不需要预训练权重

**练习 2（填空）**：YOLOv10n 中的 `n` 表示 ______，它是 5 个版本中速度 ______、精度 ______ 的版本。

**练习 3（简答）**：如果你需要在树莓派这种算力很弱的设备上做实时检测，应该选 `n/s/m/l/x` 中的哪个？为什么？

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/01.03_yolo_inference/answers_quickstart.txt



---

## 第二部分：单张图片推理与结果解读


In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

# 加载模型（如果上一节已加载可跳过）
model = YOLO("yolov10n.pt", task='detect')

# 读取测试图片（YOLO 默认 COCO 数据集训练，能识别 80 类常见物体，但"水果"未必在内）
img_path = "./images/sample_fruit.jpg"
img = cv2.imread(img_path)
print("图片形状:", img.shape)

# 推理
results = model(source=img_path, save=True, conf=0.05)
print("\n✅ 推理完成")


## 1. 解读检测结果 `results`

`results` 是一个列表（每张图片对应一个元素），核心信息都在 `results[0]`：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">属性</th><th align="left">含义</th><th align="left">示例</th></tr>
<tr><td align="left"><code>results[0].boxes</code></td><td align="left">所有检测到的边界框</td><td align="left">多个目标的集合</td></tr>
<tr><td align="left"><code>boxes.xyxy</code></td><td align="left">边界框坐标 (x1,y1,x2,y2)</td><td align="left">左上角+右下角</td></tr>
<tr><td align="left"><code>boxes.conf</code></td><td align="left">每个框的置信度 (0-1)</td><td align="left">0.92 表示92%确定</td></tr>
<tr><td align="left"><code>boxes.cls</code></td><td align="left">每个框的类别编号</td><td align="left">0=person, 1=bicycle...</td></tr>
<tr><td align="left"><code>results[0].names</code></td><td align="left">类别编号→名称的映射</td><td align="left">{0:'person', ...}</td></tr>
</table>

In [ ]:
# 解读检测结果
result = results[0]
boxes = result.boxes

print(f"检测到 {len(boxes)} 个目标\n")
print("类别映射表（前 10 类）:", dict(list(result.names.items())[:10]))
print()

if len(boxes) > 0:
    for i, box in enumerate(boxes):
        xyxy = box.xyxy[0].tolist()         # 边界框 [x1, y1, x2, y2]
        conf = box.conf[0].item()           # 置信度
        cls = int(box.cls[0].item())        # 类别编号
        name = result.names[cls]            # 类别名称
        print(f"目标 {i+1}: 类别={name}(id={cls}), 置信度={conf:.3f}, 框={xyxy}")
else:
    print("未检测到目标。可能是：")
    print("  1. 图片中物体不在 COCO 80 类中（如具体的水果种类）")
    print("  2. 置信度阈值 conf 设置过高")
    print("  3. 物体太小或模糊")
    print("\n这正是第 3 章「迁移训练」要解决的问题——教 YOLO 识别自定义类别！")


## 2. 可视化检测结果

YOLO 提供了 `result.plot()` 方法，自动在原图上画出边界框和标签（内部就是用第 1 章学的 OpenCV 绘图函数）：

In [ ]:
# result.plot() 返回带标注的图像（BGR 格式）
annotated = result.plot()

# ⚠️ 关键：plot() 返回的是 BGR 图，matplotlib 需要 RGB，必须转换！
annotated_rgb = annotated[:, :, ::-1]   # BGR → RGB 的快捷写法（反转通道维度）

plt.figure(figsize=(12, 8))
plt.imshow(annotated_rgb)
plt.title("YOLO Detection Visualization")
plt.axis('off')
plt.show()


> 💡 `annotated[:, :, ::-1]` 是 `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` 的 NumPy 等价写法——反转最后一维（通道维），BGR 就变成了 RGB。

## 3. 关键推理参数

`model()` 支持很多参数控制推理行为：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">参数</th><th align="left">默认值</th><th align="left">含义</th></tr>
<tr><td align="left"><code>conf</code></td><td align="left">0.25</td><td align="left">置信度阈值，低于此值的检测结果被过滤。调低→更多结果但有误检；调高→更准但可能漏检</td></tr>
<tr><td align="left"><code>imgsz</code></td><td align="left">640</td><td align="left">推理时图片缩放到的尺寸。越大越准但越慢</td></tr>
<tr><td align="left"><code>save</code></td><td align="left">False</td><td align="left">是否保存结果图到 runs/detect/ 目录</td></tr>
<tr><td align="left"><code>save_txt</code></td><td align="left">False</td><td align="left">是否保存检测结果为 txt（YOLO 标注格式）</td></tr>
</table>

我们对比不同 `conf` 阈值的效果：

In [ ]:
# 对比不同置信度阈值
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, conf_val in zip(axes, [0.5, 0.1, 0.01]):
    res = model(source=img_path, conf=conf_val, verbose=False)
    annotated = res[0].plot()[:, :, ::-1]
    ax.imshow(annotated)
    ax.set_title(f"conf={conf_val} (detected: {len(res[0].boxes)})")
    ax.axis('off')

plt.suptitle("Confidence Threshold (conf) Comparison", fontsize=14)
plt.tight_layout()
plt.show()


可以看到：`conf` 越低，检测到的目标越多（但可能包含误检）。实际应用中需要根据场景权衡。

---

## 本节练习

**练习 1（选择）**：`result.boxes.conf[0]` 返回的值是 0.85，表示什么？
- A. 模型有 85% 的概率出错
- B. 模型对这个检测结果有 85% 的把握
- C. 边界框的面积占图片的 85%
- D. 这个目标在图片中的位置

**练习 2（填空）**：`result.plot()` 返回的图像是 ______ 色彩格式（BGR/RGB），用 matplotlib 显示前需要转换成 ______ 格式。

**练习 3（代码）**：写一段代码，对 `images/sample_wangzai.png` 做推理，**只打印置信度大于 0.5 的目标**，并把结果可视化。

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/01.03_yolo_inference/answers_inference.txt



---

## 第三部分：实时推理（摄像头与视频）

> ⚠️ 摄像头实时推理需要本地硬件，云环境无摄像头。云环境可用视频文件替代。


In [ ]:
# ⚠️ 本代码需要本地摄像头，云环境无法运行
# 本地运行时取消下方注释

# from ultralytics import YOLO
# import cv2
# import time
#
# # 加载 YOLO 模型
# print("正在加载 YOLO 模型...")
# model = YOLO("yolov10n.pt")
# print("模型加载完成！")
#
# # 打开摄像头（0 表示默认摄像头）
# cap = cv2.VideoCapture(0)
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)    # 设置分辨率宽
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)   # 设置分辨率高
#
# if not cap.isOpened():
#     print("❌ 无法打开摄像头，请检查设备连接")
# else:
#     print("✅ 摄像头已打开，按 q 键退出检测窗口")
#     while True:
#         ret, frame = cap.read()          # 读取一帧
#         if not ret:
#             print("无法读取画面")
#             break
#
#         # 对当前帧做 YOLO 检测
#         results = model(source=frame, conf=0.4, verbose=False)
#         annotated = results[0].plot()    # 画出检测结果
#
#         # 显示画面（cv2.imshow 在 Jupyter 中不弹窗，需在本地终端运行）
#         cv2.imshow("YOLO Real-time Detection", annotated)
#
#         # 按 q 键退出（每帧等待 1 毫秒检测按键）
#         if cv2.waitKey(1) & 0xFF == ord('q'):
#             break
#
#     # 释放资源
#     cap.release()
#     cv2.destroyAllWindows()

print("⚠️ 本代码需本地摄像头。云环境请看下方 2.3 视频文件方案。")
print("本地运行时请取消上方注释，在终端执行（非 Jupyter）。")


### 关键 API 解释

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">API</th><th align="left">作用</th></tr>
<tr><td align="left"><code>cv2.VideoCapture(0)</code></td><td align="left">打开默认摄像头（参数 0 是摄像头编号，多个摄像头时为 0/1/2）</td></tr>
<tr><td align="left"><code>cap.read()</code></td><td align="left">读取一帧画面，返回 (是否成功, 帧数据)</td></tr>
<tr><td align="left"><code>cv2.imshow(win_name, img)</code></td><td align="left">在窗口显示图片（Jupyter 内不弹窗，需终端运行）</td></tr>
<tr><td align="left"><code>cv2.waitKey(1)</code></td><td align="left">等待 1 毫秒检测按键，<code>& 0xFF == ord('q')</code> 判断是否按了 q</td></tr>
<tr><td align="left"><code>cap.release()</code></td><td align="left">释放摄像头资源（必须调用，否则摄像头一直被占用）</td></tr>
</table>

## 1. 命令行一键调用（需本地环境）

Ultralytics 提供了命令行工具 `yolo`，无需写 Python 代码就能检测：

In [ ]:
# 命令行方式：打开摄像头实时检测（需本地环境）
# 直接在终端运行，不要在 Jupyter 中运行

# yolo predict detect model=yolov10n.pt source=0 show

# 参数说明：
#   predict      子命令：预测
#   detect       任务类型：检测
#   model=...    模型权重
#   source=0     数据源：0 表示摄像头（也可填图片/视频路径）
#   show         实时显示结果窗口

print("此命令需在本地终端运行，不在 Jupyter 中执行")
print("示例: yolo predict detect model=yolov10n.pt source=0 show")


## 2. 视频文件检测（云环境可用）

云环境虽然没有摄像头，但可以处理**视频文件**——逻辑完全一样，只是把 `VideoCapture(0)` 改成 `VideoCapture("视频路径")`。

> 💡 下面用 **多张图片模拟视频帧** 来演示逐帧检测流程（避免下载外网视频）。
> 如果你有自己的 mp4 文件，把 `source` 换成视频路径即可（如 `"my_video.mp4"`）。

In [ ]:
# 批量处理多张图片（模拟视频逐帧检测，云环境可用）
# YOLO 的 source 支持传入"图片目录"，会自动逐张处理（等价于视频逐帧）
from ultralytics import YOLO
import os, glob

model = YOLO("yolov10n.pt")

# 用 images/ 目录下的 jpg 图片作为"视频帧"
img_dir = "./images/"
frame_files = sorted(glob.glob(os.path.join(img_dir, "*.jpg")) + glob.glob(os.path.join(img_dir, "*.JPG")))
print(f"找到 {len(frame_files)} 张图片作为视频帧")

if frame_files:
    # 逐张推理（YOLO 会把整个目录当作"视频"处理，结果存到 runs/detect/）
    results = model(source=img_dir, save=True, conf=0.3, verbose=False)
    print("✅ 批量处理完成！")
    print("结果保存在 runs/detect/ 目录下")
    print(f"共处理 {len(results)} 张图片（帧）")
else:
    print("⚠️ 未找到图片，请确认 ./images/ 目录下有 jpg 文件")

> 💡 实际应用中，把 `source` 换成本地视频文件路径（如 `"my_video.mp4"`）即可处理自己的视频。处理结果会保存为带检测框的视频文件。

## 3. 性能提示

实时检测的帧率（FPS）取决于：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">因素</th><th align="left">影响</th><th align="left">优化建议</th></tr>
<tr><td align="left">模型大小</td><td align="left">n 比 x 快 10 倍以上</td><td align="left">实时场景用 n 或 s</td></tr>
<tr><td align="left">图像尺寸 imgsz</td><td align="left">越小越快</td><td align="left">实时可用 320 或 416</td></tr>
<tr><td align="left">硬件</td><td align="left">GPU 比 CPU 快 5-20 倍</td><td align="left">有 GPU 时务必用 GPU</td></tr>
<tr><td align="left">conf 阈值</td><td align="left">越低每帧检测越多，略慢</td><td align="left">实时用 0.4-0.5</td></tr>
</table>

---

## 本节练习

**练习 1（选择）**：在 CANNLab 云环境中，如何体验"实时检测"的效果？
- A. 直接调用 `cv2.VideoCapture(0)` 即可
- B. 云环境无摄像头，只能用视频文件 `cv2.VideoCapture("xxx.mp4")` 替代
- C. 实时检测在云上无法实现
- D. 需要特殊权限才能用摄像头

**练习 2（填空）**：实时检测循环中，退出通常通过检测按键 `q`，对应的代码是 `if cv2.waitKey(1) & 0xFF == ______`。

**练习 3（简答）**：为什么 `cap.release()` 必须在循环结束后调用？如果不调用会有什么后果？

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/01.03_yolo_inference/answers_realtime.txt
